# Lughaat-TTS-82M — training session

Runs the next units of work (data prep → Stage 1 → Stage 2 → export → upload) and stops cleanly before the 12 h limit. **Re-run this notebook (Save Version → Save & Run All) until it reports `done=True`.**

One-time setup (about 10 minutes):
1. Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**.
2. **Add-ons → Secrets**: add `HF_TOKEN` = your Hugging Face token with *write* scope, and tick it so it is attached to this notebook.
   If the GitHub repo is private, also add `GITHUB_TOKEN` = a GitHub personal access token with read access to the repo.
3. Accept the terms of the gated dataset at https://huggingface.co/datasets/ai4bharat/Rasa while logged in with the same Hugging Face account.

The code is cloned from GitHub (`PROJECT_GIT_URL` below). Everything else (repos, data, checkpoints, state) is created in your Hugging Face account: `<user>/lughaat-tts-work` (private working state) and `<user>/lughaat-tts-82m` (the model).


In [ ]:
# ===================== configuration (edit if you like) =====================
HF_USERNAME = "muhammadnoman76"   # HF account/org that owns the repos (default = owner of HF_TOKEN)
PROJECT_GIT_URL = "https://github.com/MuhammadNoman76/lughaat-tts"   # the code; GITHUB_TOKEN secret is used if the repo is private
SESSION_BUDGET_MIN = 675     # stop cleanly after this many minutes (Kaggle limit is 12 h)
URDU_HOURS_PER_SPEAKER = 6   # Rasa hours per speaker (neutral styles first); raise for more data / longer training
ENGLISH_REPLAY_HOURS = 3     # LibriTTS-R hours mixed in so English sounds are not forgotten (never below 3)
STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 5
SHARD_MINUTES = 60           # one shard = one checkpoint + backup; a killed session loses at most this much
REPO_PRIVATE = 1             # 1 = private HF repos (flip to public later in the HF UI)
USE_INDICVOICES_R = 0        # 1 = also add IndicVoices-R Urdu (needs its gated terms accepted)
LLM_LEXICON_BUDGET_USD = 0   # >0 and an ANTHROPIC_API_KEY secret = LLM pronunciation candidates for the lexicon
FORCE_PHASE = ""             # advanced: jump to a phase, e.g. "stage2"
# ============================================================================
import os, json, shutil, subprocess, sys, glob
for k, v in dict(HF_USERNAME=HF_USERNAME, SESSION_BUDGET_MIN=SESSION_BUDGET_MIN, URDU_HOURS_PER_SPEAKER=URDU_HOURS_PER_SPEAKER,
                 ENGLISH_REPLAY_HOURS=ENGLISH_REPLAY_HOURS, STAGE1_EPOCHS=STAGE1_EPOCHS, STAGE2_EPOCHS=STAGE2_EPOCHS,
                 SHARD_MINUTES=SHARD_MINUTES, REPO_PRIVATE=REPO_PRIVATE, USE_INDICVOICES_R=USE_INDICVOICES_R,
                 LLM_LEXICON_BUDGET_USD=LLM_LEXICON_BUDGET_USD, FORCE_PHASE=FORCE_PHASE).items():
    if v not in ("", None):
        os.environ[k] = str(v)
# secrets -> environment (HF_TOKEN, optional ANTHROPIC_API_KEY)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for name in ("HF_TOKEN", "GITHUB_TOKEN", "ANTHROPIC_API_KEY", "HF_USERNAME"):
        try:
            val = _s.get_secret(name)
            if val:
                os.environ[name] = val
        except Exception:
            pass
except Exception as e:
    print('kaggle_secrets unavailable:', e)
assert os.environ.get('HF_TOKEN'), 'Add the HF_TOKEN secret (Add-ons -> Secrets) and attach it to this notebook.'
print('HF_TOKEN present; GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

In [ ]:
# ===================== get the project code =====================
PROJECT = '/kaggle/working/project'
if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT)
if PROJECT_GIT_URL:
    url = PROJECT_GIT_URL
    gh = os.environ.get('GITHUB_TOKEN')
    if gh and url.startswith('https://github.com/'):
        url = url.replace('https://github.com/', f'https://{gh}@github.com/')   # private repo
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', url, PROJECT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit('git clone failed: ' + r.stderr.replace(gh or 'xxxx', '***')[-500:] +
                         '
If the repo is private: add a GITHUB_TOKEN secret (Add-ons -> Secrets) or make the repo public.')
else:
    cands = [d for d in glob.glob('/kaggle/input/*') if os.path.exists(os.path.join(d, 'kaggle', 'train_session.py'))]
    if not cands:  # zipped upload or nested folder
        for z in glob.glob('/kaggle/input/*/*.zip'):
            shutil.unpack_archive(z, '/kaggle/working/unzipped')
        cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/working/unzipped/**/kaggle/train_session.py', recursive=True)]
        cands += [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/*/**/kaggle/train_session.py', recursive=True)]
    assert cands, 'Project code not found: attach the lughaat-tts-code dataset or set PROJECT_GIT_URL'
    shutil.copytree(cands[0], PROJECT)
os.environ['PROJECT_DIR'] = PROJECT
print('project at', PROJECT, '->', sorted(os.listdir(PROJECT)))
# ===================== dependencies (staged, verified) =====================
# torch / transformers / accelerate come with the Kaggle image. A single `pip install -r` can fail as a
# whole when the resolver hits one conflict, so critical packages are installed first and verified.
subprocess.run('apt-get -qq install -y espeak-ng libespeak-ng1 libsndfile1 >/dev/null 2>&1 || true', shell=True)
PIP = [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check']
subprocess.run(PIP + ['--upgrade', 'pip'], capture_output=True, text=True)
logs = []
def pip(*pkgs, no_deps=False):
    cmd = PIP + (['--no-deps'] if no_deps else []) + list(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    logs.append((pkgs, r.returncode, (r.stdout + r.stderr)[-3000:]))
    return r.returncode == 0

# stage 1: Kokoro + misaki (critical). Fallback: no-deps + the runtime deps misaki[en] actually needs
# (spacy-curated-transformers is only for the trf tagger, which this project never uses).
ok = pip('kokoro>=0.9.4', 'misaki[en]>=0.9.4')
if not ok:
    print('pip resolver failed for kokoro/misaki[en]; retrying without dependency resolution')
    pip('kokoro>=0.9.4', 'misaki>=0.9.4', no_deps=True)
    pip('addict', 'regex', 'num2words', 'spacy', 'phonemizer-fork', 'espeakng-loader', 'loguru', 'huggingface_hub')
# stage 2: everything else, in independent groups
pip('datasets>=3.0', 'soundfile', 'librosa', 'soxr', 'jiwer', 'pyyaml', 'pytest', 'huggingface_hub>=0.25')
pip('speechmos', 'onnxruntime', 'onnx', 'onnxconverter-common')
pip('speechbrain')
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=False)

# verify the imports the pipeline cannot live without; stop here with the pip logs if one is missing
missing = []
for mod in ('kokoro', 'misaki', 'misaki.en', 'datasets', 'soundfile', 'librosa', 'soxr', 'huggingface_hub', 'yaml', 'pytest'):
    r = subprocess.run([sys.executable, '-c', f'import {mod}'], capture_output=True, text=True)
    if r.returncode != 0:
        missing.append((mod, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ''))
for pkgs, rc, out in logs:
    if rc != 0:
        print(f'--- pip FAILED for {pkgs} ---\n{out}\n')
if missing:
    raise SystemExit('Required Python packages are missing after installation: ' + str(missing) +
                     '\nSee the pip output above. Fix the environment (or pin versions in requirements-kaggle.txt) and re-run.')
print('dependencies OK:', subprocess.run([sys.executable, '-c',
      'import kokoro, misaki, datasets, torch; print("kokoro", kokoro.__version__, "misaki", misaki.__version__ if hasattr(misaki, "__version__") else "", "datasets", datasets.__version__, "torch", torch.__version__)'],
      capture_output=True, text=True).stdout.strip())


In [ ]:
# ===================== run the session =====================
# Idempotent: downloads state.json from <user>/lughaat-tts-work, runs the next shards/phases, backs up after every
# shard, and exits before the Kaggle limit. Re-run this notebook until the summary says done=True.
rc = subprocess.call([sys.executable, '-u', os.path.join(PROJECT, 'kaggle', 'train_session.py')], cwd=PROJECT)
print('exit code', rc)
print(open('/kaggle/working/summary.txt').read() if os.path.exists('/kaggle/working/summary.txt') else 'no summary')
if os.path.exists('/kaggle/working/state.json'):
    st = json.load(open('/kaggle/working/state.json'))
    print(json.dumps({k: st.get(k) for k in ('phase', 'done', 'best_s1_epoch', 'best_s2_epoch', 'stop_reason', 'last_error')}, indent=1, default=str))